# NB09 — Pico Sweep TDE: Laufzeitdifferenz am Stahlstab

## Versuchsaufbau

### Vorversuch (2026-06-29) — 2 Sensoren, kein CH1-Trigger

```
Raspberry Pi Pico
  GP15 (Pin 20) ──── 100 Ω ──── Piezo (+)     (Frequenzsweep 20 Hz → 100 kHz)
  GP14 (Pin 19)  [Kabel unterbrochen → kein Frequenzstempel verfügbar]

Rigol DS1104Z
  CH1: deaktiviert
  CH2: deaktiviert (defekt)
  CH3: LM358-Sensor A  (Stahlstab, Probe ×10, 0.5 V/div)
  CH4: LM358-Sensor B  (Stahlstab, Probe ×10, 0.5 V/div)
  Trigger: AUTO, keine feste Frequenzreferenz
```

### Folgeversuch (geplant) — 3 Sensoren, CH1 als dritter Sensor

```
Raspberry Pi Pico
  GP15 (Pin 20) ──── 100 Ω ──── Piezo (+)
  GP14 (Pin 19)  [optional: Frequenzstempel; sonst frei lassen]

Rigol DS1104Z
  CH1: LM358-Sensor C  (neuer Messpunkt, AC, ×10, 0.5 V/div)
  CH2: deaktiviert (defekt)
  CH3: LM358-Sensor A  (Stahlstab, wie Vorversuch)
  CH4: LM358-Sensor B  (Stahlstab, wie Vorversuch)
  Trigger: AUTO auf CH3 (stärkstes Signal, 50 mV Level)
```

## Pico Sweep-Parameter (Rust-Quellcode `pico_rust_sweep/src/main.rs`)

| Parameter | Wert | Bedeutung |
|---|---|---|
| F_START | 20 Hz | Startfrequenz |
| F_STOP | 100 000 Hz | Endfrequenz |
| SWEEP_US | 5 000 000 µs | 5 Sekunden pro Sweep-Zyklus |
| MARKER_STEP_HZ | 5 000 Hz | Puls auf GP14 alle 5 kHz |
| MARKER_PULSE_US | 500 µs | Pulsbreite des Markers |
| Sweep-Rate | 20 kHz / s | linear, daraus abgeleitet |

**Beobachtung**: Der Stahlstab hat eine Eigenfrequenz bei ~8 750 Hz.
Diese dominiert alle Captures unabhängig von der aktuellen Pico-Frequenz.
Alle TDE-Messungen beziehen sich daher auf diese Modalfrequenz.


In [ ]:
%matplotlib inline
import sys, time, json, math
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags

CHAR_ROOT = Path('..').resolve()
sys.path.insert(0, str(CHAR_ROOT))

# ── Pico sweep constants (match Rust source) ──────────────────────────────────
PICO_F_START_HZ    = 20
PICO_F_STOP_HZ     = 100_000
PICO_SWEEP_S       = 5.0
PICO_SWEEP_RATE    = (PICO_F_STOP_HZ - PICO_F_START_HZ) / PICO_SWEEP_S  # 20 kHz/s

# ── Scope acquisition constants ───────────────────────────────────────────────
MAX_PTS    = 50_000    # sicheres Limit DS1104Z per VISA-Transfer
RUN_WAIT_S = 1.0       # RUN-Zeit vor STOP (AUTO-Trigger)

# ── Pilot-Daten (Vorversuch 2026-06-29) ──────────────────────────────────────
PILOT_JSON = CHAR_ROOT / 'data' / 'pico_tde_xcorr' / '20260629_170951' / 'tde_result.json'

print(f'CHAR_ROOT   : {CHAR_ROOT}')
print(f'Pilot-Daten : {PILOT_JSON.exists()}  →  {PILOT_JSON}')


---
## Teil A — Auswertung Vorversuch (2 Sensoren, 2026-06-29)

**Methode**: AUTO-Trigger (RUN 1 s → STOP), 17 Frames, CH3 + CH4, MAX_PTS = 50 000.

Zwei unabhängige TDE-Schätzer:

1. **Phase-Differenz**: Phasor bei der dominanten Frequenz f₀ auslesen, dann
   τ₃₄ = −(φ₃ − φ₄) / (2π f₀)

2. **Kreuzkorrelation**: arg max der Korrelationsfunktion s₃ ⋆ s₄ innerhalb ±200 µs Suchfenster.
   Zeitauflösung = 1/f_s ≈ 0.2 µs (quantisiert).


In [ ]:
with open(PILOT_JSON) as f:
    pilot = json.load(f)

records = pilot['records']

# Vorzeichenkonvention:
# tau_ph_us im JSON = (φ₃ − φ₄)/(2π f)  — nicht physikalisch
# physikalisches TDE τ₃₄ = t_Ankunft_CH3 − t_Ankunft_CH4 = −tau_ph_us
tau_ph = np.array([-r['tau_ph_us'] for r in records])
tau_xc = np.array([ r['tau_xc_us'] for r in records])
freqs  = np.array([ r['f']          for r in records])

# Nur Stahlresonanz-Frames (8–10 kHz), keine Obertöne
clean = (freqs >= 8_000) & (freqs <= 10_000)
tau_ph_c = tau_ph[clean]
tau_xc_c = tau_xc[clean]
f_res    = float(np.median(freqs[clean]))

print(f'N gesamt : {len(records)}')
print(f'N valide : {clean.sum()}  (8 – 10 kHz, Stahlresonanz)')
print(f'f_Resonanz: {f_res:.0f} Hz')
print()
print(f'Phase-Diff  τ₃₄ : {np.mean(tau_ph_c):+.4f} ± {np.std(tau_ph_c):.4f} µs')
print(f'XCorr       τ₃₄ : {np.median(tau_xc_c):+.2f} µs  (Auflösung ≈ 0.20 µs)')
print()
print('τ₃₄ > 0  →  CH3 trifft später ein als CH4  →  Quelle näher an CH4')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('TDE Vorversuch — CH3 vs. CH4, Stahlstab (2026-06-29)', fontsize=11)

# Links: Frame-für-Frame
ax = axes[0]
frame_idx = np.where(clean)[0] + 1
ax.scatter(frame_idx, tau_ph_c, s=45, color='steelblue', zorder=3,
           label='Phase-Diff (sub-sample)')
m, s = np.mean(tau_ph_c), np.std(tau_ph_c)
ax.axhline(m, color='steelblue', lw=2, label=f'Mittel = {m:+.3f} µs')
ax.fill_between([frame_idx[0]-0.5, frame_idx[-1]+0.5], m-s, m+s,
                alpha=0.18, color='steelblue', label=f'±σ = {s:.3f} µs')
ax.axhline(np.median(tau_xc_c), color='darkorange', ls='--', lw=1.8,
           label=f'XCorr Median = {np.median(tau_xc_c):+.2f} µs')
ax.set_xlabel('Frame-Index')
ax.set_ylabel('τ₃₄  [µs]')
ax.set_title('Frame-für-Frame TDE (Stahlresonanz-Frames)')
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3)

# Rechts: Histogramm
ax = axes[1]
ax.hist(tau_ph_c, bins=9, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(m, color='steelblue', lw=2.2, label=f'µ = {m:+.3f} µs')
ax.axvline(m-s, color='steelblue', lw=1, ls=':')
ax.axvline(m+s, color='steelblue', lw=1, ls=':')
ax.axvline(np.median(tau_xc_c), color='darkorange', ls='--', lw=2,
           label=f'XCorr = {np.median(tau_xc_c):+.2f} µs')
ax.set_xlabel('τ₃₄  [µs]')
ax.set_ylabel('Häufigkeit')
ax.set_title('Verteilung Phase-Diff TDE')
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_png = PILOT_JSON.parent / 'tde_plot.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot gespeichert: {out_png}')


In [ ]:
tau_best_s = np.mean(tau_ph_c) * 1e-6

print(f'τ₃₄ = {tau_best_s*1e6:+.4f} µs  (Phase-Diff, N={clean.sum()}, ±{np.std(tau_ph_c)*1e3:.1f} ns)')
print()
print(f'Wegdifferenz Δd = |τ₃₄| × v:')
print(f'  {"Material":<20}  {"v [m/s]":>8}   {"Δd [mm]":>8}')
print(f'  {"-"*42}')
for mat, v in [('Stahl (längs)', 5000), ('Aluminium (längs)', 6400),
               ('Granit', 3000), ('Beton', 3500),
               ('Holz (Faser)', 4000), ('Luft', 343)]:
    print(f'  {mat:<20}  {v:>8}   {abs(tau_best_s)*v*1000:>8.2f} mm')


---
## Teil B — Wiederholung mit 3 Sensoren

**Ziel**: CH1 wird als dritter akustischer Sensor angeschlossen.
Mit drei Laufzeitdifferenzen (CH1−CH3, CH1−CH4, CH3−CH4) kann die
Quellposition eindimensional auf dem Stab rekonstruiert werden.

### Scope-Konfiguration für den Folgeversuch

| Kanal | Belegung | Kopplung | Tastkopf | Skala |
|---|---|---|---|---|
| CH1 | LM358-Sensor C | AC | ×10 | 0.5 V/div |
| CH2 | — deaktiviert — | — | — | — |
| CH3 | LM358-Sensor A | AC | ×10 | 0.5 V/div |
| CH4 | LM358-Sensor B | AC | ×10 | 0.5 V/div |
| Trigger | AUTO, Edge CH3 | — | — | 50 mV Level |

**Wichtig (DS1104Z Eigenheit)**:
Bei drei aktiven Kanälen und Tiefenspeicher kann CH3 oder CH4 gelegentlich
0 Byte zurückgeben. Lese-Reihenfolge deshalb immer: **zuerst CH4, dann CH3, zuletzt CH1**.

### Sensor-Positionen ausmessen
Vor dem Versuch die Abstände aller drei Sensoren vom Piezo-Auflagepunkt
mit einem Maßband messen und in `SENSOR_POS_MM` eintragen.


In [ ]:
from scope.config import load_config
from scope.instrument import InstrumentConnection, acquire_waveform_bytes
from plant_ae.watering import CONFIG_PATH

# ── 3-Sensor Konfiguration ───────────────────────────────────────────────────
SENSOR_CHANNELS = (1, 3, 4)
CH_SCALE = {1: 0.5,  3: 0.5,  4: 0.5}
CH_PROBE = {1: 10,   3: 10,   4: 10 }
TIMEBASE = 0.01   # 10 ms/div → 120 ms Fenster

# Sensor-Positionen auf Stab (mm ab Piezo-Auflagepunkt) — vor Versuch ausfüllen!
SENSOR_POS_MM = {
    1: None,   # CH1-Sensor: __ mm
    3: None,   # CH3-Sensor: __ mm
    4: None,   # CH4-Sensor: __ mm
}

RUN_ID_3S = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR_3S = CHAR_ROOT / 'data' / 'pico_tde_3sensor' / RUN_ID_3S
OUT_DIR_3S.mkdir(parents=True, exist_ok=True)
config = load_config(CONFIG_PATH)

print(f'Run-ID  : {RUN_ID_3S}')
print(f'Output  : {OUT_DIR_3S}')


In [ ]:
def configure_3sensor(conn):
    conn.write(':STOP'); time.sleep(0.2)
    conn.write(':CHAN2:DISP OFF')
    for ch in SENSOR_CHANNELS:
        conn.write(f':CHAN{ch}:DISP ON')
        conn.write(f':CHAN{ch}:COUP AC')
        conn.write(f':CHAN{ch}:PROB {CH_PROBE[ch]}')
        conn.write(f':CHAN{ch}:SCAL {CH_SCALE[ch]}')
        conn.write(f':CHAN{ch}:OFFS 0')
    conn.write(f':TIM:SCAL {TIMEBASE}')
    conn.write(':TIM:OFFS 0')
    conn.write(':TRIG:MODE EDGE')
    conn.write(':TRIG:EDGE:SOUR CHAN3')
    conn.write(':TRIG:EDGE:LEV 0.05')
    conn.write(':TRIG:SWE AUTO')
    conn.write(':RUN'); time.sleep(0.5)
    print('Scope konfiguriert (3 Sensoren, AUTO-Trigger CH3).')


def acquire_3sensor_frame(conn):
    """RUN 1 s → STOP → lese CH4, CH3, CH1 (Reihenfolge: DS1104Z Deep-Memory-Bug)."""
    conn.write(':RUN'); time.sleep(RUN_WAIT_S)
    conn.write(':STOP'); time.sleep(0.2)
    v = {}; fs_val = None
    for ch in (4, 3, 1):
        pre, raw = acquire_waveform_bytes(conn, ch, mode='RAW', fmt='BYTE',
                                          max_points=MAX_PTS)
        if len(raw) > 0 and pre.xincrement > 0:
            v[ch] = (np.frombuffer(bytes(raw), dtype=np.uint8).astype(np.float32)
                     - pre.yorigin - pre.yreference) * pre.yincrement
            if fs_val is None:
                fs_val = 1.0 / pre.xincrement
    return v, fs_val


def fft_at(sig, fs_val, freq):
    """Amplitude [V] und Phase [rad] bei Frequenz freq."""
    n     = len(sig)
    freqs = np.fft.rfftfreq(n, 1/fs_val)
    spec  = np.fft.rfft(sig * np.hanning(n))
    idx   = int(np.argmin(np.abs(freqs - freq)))
    return float(np.abs(spec[idx]))*2/n, float(np.angle(spec[idx]))


def xcorr_tde(s_a, s_b, fs_val, max_lag_us=200):
    """Kreuzkorrelations-TDE: τ in µs (s_a trifft nach s_b ein wenn positiv)."""
    n       = min(len(s_a), len(s_b))
    max_lag = int(max_lag_us*1e-6*fs_val)
    corr    = correlate(s_a[:n], s_b[:n], mode='full')
    lags    = correlation_lags(n, n, mode='full')
    win     = np.abs(lags) <= max_lag
    lag     = int(lags[win][np.argmax(corr[win])])
    return lag / fs_val * 1e6   # µs


print('Helpers bereit.')


In [ ]:
N_FRAMES = 30   # Anzahl Frames — bei Bedarf erhöhen

PAIRS_DEF = [(1,3,'CH1−CH3'), (1,4,'CH1−CH4'), (3,4,'CH3−CH4')]
frame_records = []

print(f'Erfasse {N_FRAMES} Frames (3 Sensoren)\n')

with InstrumentConnection(config) as conn:
    configure_3sensor(conn)

    for i in range(N_FRAMES):
        v, fs_val = acquire_3sensor_frame(conn)
        missing = [ch for ch in SENSOR_CHANNELS if ch not in v]
        if missing:
            print(f'  Frame {i+1:2d}: kein Signal CH{missing}  → übersprungen')
            continue

        n      = min(len(v[ch]) for ch in SENSOR_CHANNELS)
        freqs  = np.fft.rfftfreq(n, 1/fs_val)
        spec3  = np.abs(np.fft.rfft(v[3][:n]*np.hanning(n)))*2/n
        mask   = (freqs>=3_000)&(freqs<=20_000)
        f_res  = float(freqs[mask][np.argmax(spec3[mask])])

        rec = {'frame': i+1, 'ts': datetime.now().isoformat(),
               'fs': fs_val, 'f_res_hz': f_res}
        for ch in SENSOR_CHANNELS:
            amp, ph = fft_at(v[ch], fs_val, f_res)
            rec[f'amp_ch{ch}'] = float(amp)
            rec[f'phase_ch{ch}'] = float(ph)
        for ca, cb, lbl in PAIRS_DEF:
            tau_xc = xcorr_tde(v[ca], v[cb], fs_val)
            tau_ph = -(rec[f'phase_ch{ca}']-rec[f'phase_ch{cb}'])/(2*math.pi*f_res)
            rec[f'tau_xc_{ca}{cb}_us'] = tau_xc
            rec[f'tau_ph_{ca}{cb}_us'] = tau_ph * 1e6
        frame_records.append(rec)

        pairs_str = '  '.join(f"τ_{a}{b}={rec[f'tau_ph_{a}{b}_us']:+.2f}µs"
                              for a,b,_ in PAIRS_DEF)
        print(f"  Frame {i+1:2d}: f={f_res:.0f} Hz  {pairs_str}")

with (OUT_DIR_3S/'records.jsonl').open('w') as fh:
    for r in frame_records: fh.write(json.dumps(r)+'\n')
print(f'\n{len(frame_records)} Frames → {OUT_DIR_3S}/records.jsonl')


---
## Teil C — Auswertung 3-Sensor-Messung

*Zelle nach der Messung ausführen — `DATA_DIR` auf das erzeugte Verzeichnis setzen.*


In [ ]:
# ── Datenpfad ─────────────────────────────────────────────────────────────────
# Nach der Messung: RUN_ID_3S durch tatsächliches Datum/Uhrzeit ersetzen
# DATA_DIR = CHAR_ROOT / 'data' / 'pico_tde_3sensor' / '20260629_XXXXXX'
DATA_DIR = OUT_DIR_3S   # automatisch gesetzt wenn Messung oben lief

PAIRS_DEF = [(1,3,'CH1−CH3'), (1,4,'CH1−CH4'), (3,4,'CH3−CH4')]
records_3s = []
with open(DATA_DIR/'records.jsonl') as fh:
    for line in fh: records_3s.append(json.loads(line))

print(f'{len(records_3s)} Frames aus {DATA_DIR.name}')
results_3s = {}
for ca, cb, lbl in PAIRS_DEF:
    tph = np.array([r[f'tau_ph_{ca}{cb}_us'] for r in records_3s])
    txc = np.array([r[f'tau_xc_{ca}{cb}_us'] for r in records_3s])
    results_3s[(ca,cb)] = {'lbl': lbl, 'tau_ph': tph, 'tau_xc': txc}
    print(f'{lbl:12s}  Phase: {np.mean(tph):+.3f} ± {np.std(tph):.3f} µs'
          f'  |  XCorr: {np.median(txc):+.2f} µs')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(f'TDE 3 Sensoren — {DATA_DIR.name}', fontsize=11)

colors = ['steelblue', 'seagreen', 'firebrick']
for ax, (ca, cb, lbl), col in zip(axes, PAIRS_DEF, colors):
    tph = results_3s[(ca,cb)]['tau_ph']
    m, s = np.mean(tph), np.std(tph)
    ax.scatter(range(1, len(tph)+1), tph, s=30, color=col, zorder=3)
    ax.axhline(m, color=col, lw=2, label=f'µ={m:+.3f} µs')
    ax.fill_between([0, len(tph)+1], m-s, m+s, alpha=0.15, color=col)
    ax.axhline(np.median(results_3s[(ca,cb)]['tau_xc']), ls='--', color='k',
               lw=1.2, label=f'xcorr={np.median(results_3s[(ca,cb)]["tau_xc"]):+.2f} µs')
    ax.set_title(lbl); ax.set_xlabel('Frame'); ax.set_ylabel('τ [µs]')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR/'tde_3sensor_plot.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Teil D — 1D-TDOA-Ortung (kollineare Sensoren)

Mit drei kollinear auf dem Stab platzierten Sensoren lässt sich die
Quellposition aus zwei unabhängigen Laufzeitdifferenzen bestimmen.

**Modell (1D, Stab in x-Richtung)**:

Sei $x_i$ die Position von Sensor $i$ (m, ab Piezo-Ende). Quelle bei $x_s$, Schallgeschwindigkeit $v$.

$$\tau_{ij} = \frac{|x_s - x_i| - |x_s - x_j|}{v}$$

Überbestimmtes System (3 Paare, 1 Unbekannte $x_s$) → Kleinste-Quadrate-Minimierung.


In [ ]:
from scipy.optimize import minimize_scalar

V_STEEL = 5000   # m/s (Stahlstab, Longitudinalwelle)

if any(v is None for v in SENSOR_POS_MM.values()):
    print('⚠  SENSOR_POS_MM nicht ausgefüllt — Ortung übersprungen.')
    print('   Werte vor der Messung in der Konfigurations-Zelle eintragen.')
else:
    pos = {ch: p*1e-3 for ch, p in SENSOR_POS_MM.items()}   # mm → m

    # Mittleres τ aus Phase-Diff für alle Paare
    tau_obs = {(ca,cb): np.mean(results_3s[(ca,cb)]['tau_ph'])*1e-6
               for ca,cb,_ in PAIRS_DEF}

    def residual(xs):
        r = 0.0
        for (ca, cb), tau in tau_obs.items():
            tau_pred = (abs(xs - pos[ca]) - abs(xs - pos[cb])) / V_STEEL
            r += (tau_pred - tau)**2
        return r

    lo = min(pos.values()) - 0.05
    hi = max(pos.values()) + 0.05
    res = minimize_scalar(residual, bounds=(lo, hi), method='bounded')
    xs_mm = res.x * 1000

    print(f'Geschätzte Quellposition: {xs_mm:.1f} mm  (ab Piezo-Ende, v={V_STEEL} m/s)')
    print(f'Residuum: {res.fun:.2e} s²')
    for ch, p in pos.items():
        print(f'  CH{ch} bei {p*1000:.1f} mm  →  Abstand zur Quelle: {abs(res.x - p)*1000:.1f} mm')


---
## Zusammenfassung

| Größe | Vorversuch (2 Sensoren) | Folgeversuch (3 Sensoren) |
|---|---|---|
| Datum | 2026-06-29 | — (offen) |
| Sensoren | CH3, CH4 | CH1, CH3, CH4 |
| Trigger | AUTO, kein Marker | AUTO CH3, 50 mV |
| Frames (valide) | 15 / 17 | — |
| f_Resonanz Stahl | ~8 750 Hz | — |
| τ₃₄ Phase-Diff | **+1.95 ± 0.03 µs** | — |
| τ₃₄ XCorr | +1.60 µs (±0.20 µs Aufl.) | — |
| Δd Stahl (5 000 m/s) | **9.75 mm** | — |
| TDOA-Ortung | nicht möglich (nur 1 Paar) | geplant |

**Ergebnis Vorversuch**: CH3 trifft 1.95 µs nach CH4 ein.
Die Schallquelle (Piezo) liegt ca. 9.8 mm näher an CH4 als an CH3.
Die Messung ist hochreproduzierbar (σ = 0.03 µs = 30 ns über 15 Frames).

**Nächster Schritt**: Mit CH1 als drittem Sensor liegen drei Sensor-Paare vor,
sodass die 1D-Quellposition eindeutig rekonstruiert werden kann.
